# Naive GAIL on PointMaze Medium

In [1]:
import random
import copy
import torch
import pickle
import os
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import PointMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import *
from causal_rl.algo.imitation.gail.causal_gail import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 1
hidden_dims = {'L'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, L hidden
train_env = PointMazePCH(num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, L hidden
eval_env = PointMazePCH(num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = PointMazePCH(num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
naive_Z_sets = {}
for Xi in X:
    i = int(Xi[1:])
    cond = set()

    for j in range(i+1):
        cond.update({f'{o}{j}' for o in list(set(obs_prefix) - {'X'})})

    for j in range(i):
        cond.add(f'X{j}')
    naive_Z_sets[Xi] = cond

naive_Z_sets['X1']

{'P0', 'P1', 'W0', 'W1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj_pointmed.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 577619 trajectories


In [8]:
dims = {
    'P': 2,
    # 'L': 2,
    'W': 2,
    'X': 2
}

In [9]:
sample_obs = records[0]['obs']

# Trim Z-sets to the lookback window (this matches what you do for BC)
naive_Z_trim = trim_Z_sets(naive_Z_sets, lookback=lookback)

# Build windowed encoders that depend on relative lags (not absolute time)
naive_encode, naive_z_dim, naive_slots = build_windowed_z_encoder(
    naive_Z_trim,
    dims=dims,
    lookback=lookback,
)

naive_z_dim

10

In [10]:
# precompute expert batches once (so one_training_round doesn't redo this every time)
Z_e_naive, A_e_naive, X_e_naive = make_expert_batch(records, naive_encode)
X_e_naive = X_e_naive.to(device)

## Hyperparameters

In [11]:
# PPO
gail_gamma          = 0.99
gae_lambda          = 0.95
ppo_clip            = 0.2
ppo_epochs          = 4
ppo_minibatch_size  = 1024
entropy_coeff       = 1e-2
value_coeff         = 0.5
max_grad_norm       = 0.5
normalize_adv       = True

# discriminator
d_loss_type         = 'bce'
gp_lambda           = 5.0
d_updates           = 2
d_minibatch_size    = 1024
use_gp              = True
instance_noise_std  = 0.0
label_smoothing     = 0.0

# rollout
max_steps_per_episode   = num_steps
episodes_per_round      = 20
num_rounds_naive_gail  = 500

# network
hidden_size_actor   = 256
hidden_size_critic  = 256
hidden_size_disc    = 256
actor_lr            = 1e-4
critic_lr           = 3e-4
disc_lr             = 3e-4
num_blocks_actor    = 3
dropout_actor       = 0.05
layernorm_actor     = True

## Network Initialization

In [12]:
action_dim = train_env.env.action_space.shape[0]
action_low = float(train_env.env.action_space.low.min())
action_high = float(train_env.env.action_space.high.max())

naive_actor = ContinuousActor(
    num_inputs=naive_z_dim,
    num_outputs=action_dim,
    hidden_size=hidden_size_actor,
    std=0.0,
    action_low=action_low,
    action_high=action_high,
    num_blocks=num_blocks_actor,
    dropout=dropout_actor,
    layernorm=layernorm_actor,
).to(device)

naive_critic = Critic(
    num_inputs=naive_z_dim,
    hidden_size=hidden_size_critic,
).to(device)

naive_disc = Discriminator(
    num_inputs=naive_z_dim + action_dim,
    hidden_size=hidden_size_disc,
    dropout=0.2,
).to(device)

actor_optim_naive = torch.optim.Adam(naive_actor.parameters(), lr=actor_lr)
critic_optim_naive = torch.optim.Adam(naive_critic.parameters(), lr=critic_lr)
disc_optim_naive = torch.optim.Adam(naive_disc.parameters(), lr=disc_lr)

## Training

In [13]:
best_return = -float('inf')
best_actor_sd = None
return_window = []
WINDOW = 20

disc_scheduler = torch.optim.lr_scheduler.StepLR(disc_optim_naive, step_size=100, gamma=0.5)

logs_naive_gail = []

for it in range(1, num_rounds_naive_gail + 1):
    stats = one_training_round(
        env=train_env,
        actor=naive_actor,
        critic=naive_critic,
        discriminator=naive_disc,
        actor_optim=actor_optim_naive,
        critic_optim=critic_optim_naive,
        discriminator_optim=disc_optim_naive,
        encode=naive_encode,
        X_e=X_e_naive,
        expert_records=None,
        gamma=gail_gamma,
        gae_lambda=gae_lambda,
        ppo_clip=ppo_clip,
        epochs=ppo_epochs,
        minibatch_size=ppo_minibatch_size,
        entropy_coeff=entropy_coeff,
        value_coeff=value_coeff,
        max_grad_norm=max_grad_norm,
        normalize_adv=normalize_adv,
        loss_type=d_loss_type,
        gp_lambda=gp_lambda,
        d_updates=d_updates,
        d_minibatch_size=d_minibatch_size,
        use_gp=use_gp,
        instance_noise_std=instance_noise_std,
        label_smoothing=label_smoothing,
        max_steps=max_steps_per_episode,
        num_episodes=episodes_per_round,
        seed=seed + 10_000 + it
    )
    logs_naive_gail.append(stats)
    disc_scheduler.step()

    # rolling average return tracking
    return_window.append(stats['avg_env_return'])
    if len(return_window) > WINDOW:
        return_window.pop(0)
    avg_ret = sum(return_window) / len(return_window)

    if avg_ret > best_return:
        best_return = avg_ret
        best_actor_sd = copy.deepcopy(naive_actor.state_dict())

    if it % 10 == 0:
        print(
            f"[Naive GAIL iter {it}] "
            f"return={stats['avg_env_return']:.2f}, "
            f"D_loss={stats['D_loss']:.3f}, "
            f"actor_loss={stats['ppo_actor_loss']:.3f}, "
            f"best_avg={best_return:.2f}"
        )

# restore best checkpoint
naive_actor.load_state_dict(best_actor_sd)
print(f"Restored best checkpoint (avg return={best_return:.2f})")

[Naive GAIL iter 10] return=-689.68, D_loss=0.642, actor_loss=-0.045, best_avg=-691.78


[Naive GAIL iter 20] return=-674.35, D_loss=1.039, actor_loss=-0.030, best_avg=-691.78


[Naive GAIL iter 30] return=-682.72, D_loss=1.242, actor_loss=-0.031, best_avg=-675.87


[Naive GAIL iter 40] return=-625.88, D_loss=1.267, actor_loss=-0.029, best_avg=-604.71


[Naive GAIL iter 50] return=-485.85, D_loss=1.292, actor_loss=0.033, best_avg=-516.77


...

[Naive GAIL iter 460] return=-333.44, D_loss=1.339, actor_loss=0.005, best_avg=-317.88


[Naive GAIL iter 470] return=-359.30, D_loss=1.344, actor_loss=0.009, best_avg=-317.88


[Naive GAIL iter 480] return=-433.67, D_loss=1.347, actor_loss=0.008, best_avg=-317.88


[Naive GAIL iter 490] return=-460.80, D_loss=1.346, actor_loss=0.006, best_avg=-317.88


[Naive GAIL iter 500] return=-399.49, D_loss=1.348, actor_loss=0.007, best_avg=-317.88
Restored best checkpoint (avg return=-317.88)


## Evaluation

In [14]:
naive_gail_policy = make_gail_policy(naive_actor, naive_encode, device=device, deterministic=True)
naive_gail_policies = make_shared_policy_dict(naive_gail_policy)

In [15]:
num_eval_eps = 100
naive_gail_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=naive_gail_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    seed=seed + 90210,
    show_progress=True
)

len(naive_gail_returns)

Starting episode 1/100...


  Episode 1 ended at step 104 (terminated: True, truncated: False).
Starting episode 2/100...


  Episode 2 ended at step 106 (terminated: True, truncated: False).
Starting episode 3/100...


  Episode 3 ended at step 1000 (terminated: False, truncated: True).
Starting episode 4/100...


  Episode 4 ended at step 1000 (terminated: False, truncated: True).
Starting episode 5/100...


...

  Episode 97 ended at step 1000 (terminated: False, truncated: True).
Starting episode 98/100...


  Episode 98 ended at step 1000 (terminated: False, truncated: True).
Starting episode 99/100...


  Episode 99 ended at step 107 (terminated: True, truncated: False).
Starting episode 100/100...


  Episode 100 ended at step 107 (terminated: True, truncated: False).
Finished collecting imitator trajectories.


57086

In [16]:
naive_gail_episode_rewards = defaultdict(float)
for rec in naive_gail_returns:
    ep = rec['episode']
    naive_gail_episode_rewards[ep] += float(rec['reward'])

naive_gail_rewards = [naive_gail_episode_rewards[e] for e in range(num_eval_eps)]
sum(naive_gail_rewards) / num_eval_eps

-433.35158485018735

## Save Model

In [17]:
# save model
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, 'ngail_pointmed.pt')

naive_gail_ckpt = {
    "state_dict": naive_actor.state_dict(),
    "z_dim": naive_z_dim,
    "action_dim": action_dim,
    "hidden_size_actor": hidden_size_actor,
    "num_blocks_actor": num_blocks_actor,
    "dropout_actor": dropout_actor,
    "layernorm_actor": layernorm_actor,
    "final_tanh": True,
    "action_bounds_low": eval_env.env.action_space.low,
    "action_bounds_high": eval_env.env.action_space.high,
    "Z_sets": naive_Z_trim,
    "dims": dims,
    "lookback": lookback,
}

torch.save(naive_gail_ckpt, MODEL_PATH)
print("Saved Naive GAIL actor to:", MODEL_PATH)

Saved Naive GAIL actor to: /home/et2842/causal/causalrl/models/ngail_pointmed.pt
